# Build FAISS Index — SigLIP — Google Colab

**Qué hace este notebook:**
1. Monta Google Drive y descomprime los pósters
2. Reconstruye metadata desde `index_metadata.csv` (HF Hub)
3. Encoda los pósters con SigLIP → vectores de 768 dims
4. Construye el índice FAISS
5. Guarda `faiss_siglip.index` e `index_metadata_siglip.csv` en Drive

**Antes de ejecutar:**
- Activar GPU: `Runtime → Change runtime type → T4 GPU`
- Tener `posters.zip` en `Mi Unidad/Recomendador_Pelis/`

In [1]:
# Celda 2 — Instalar dependencias
!pip install -q transformers torch torchvision faiss-cpu pandas Pillow huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 97.9 MB/s eta 0:00:00


In [3]:
# Celda 3 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Celda 4 — Descomprimir pósters
import zipfile
from pathlib import Path

ZIP_PATH    = "/content/drive/MyDrive/Recomendador_Pelis/posters/posters.zip"
EXTRACT_DIR = "/content/posters"
POSTERS_DIR = "/content/posters/posters"  # subcarpeta dentro del zip

Path(EXTRACT_DIR).mkdir(exist_ok=True)

print("Descomprimiendo pósters...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

posters = list(Path(POSTERS_DIR).glob("*.jpg"))
print(f"Pósters disponibles: {len(posters):,}")

Descomprimiendo pósters...
Pósters disponibles: 85,048


In [8]:
# Celda 5 — Cargar metadata desde el zip (carpeta processed)
import pandas as pd
from pathlib import Path

METADATA_PATH = "/content/posters/processed/metadata.csv"

df = pd.read_csv(METADATA_PATH)

# Remap de rutas → apuntar a los pósters descomprimidos en Colab
df["poster_path"] = df["tmdbId"].apply(
    lambda tid: f"/content/posters/posters/{int(tid)}.jpg"
)

# Filtrar solo los que existen en disco
df = df[df["poster_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print(f"Películas con póster disponible: {len(df):,}")
df.head()

Películas con póster disponible: 85,082


,movieId,tmdbId,title,genres,overview,poster_path
0,7,11860,Sabrina,"Romance, Drama, Comedy","After her return from school in Paris, a playb...",/content/posters/posters/11860.jpg
1,2,8844,Jumanji,"Adventure, Fantasy, Family",When siblings Judy and Peter discover an encha...,/content/posters/posters/8844.jpg
2,3,15602,Grumpier Old Men,"Romance, Comedy",A family wedding reignites the ancient feud be...,/content/posters/posters/15602.jpg
3,8,45325,Tom and Huck,"Family, Action, Adventure, Drama","A mischievous young boy, Tom Sawyer, witnesses...",/content/posters/posters/45325.jpg
4,5,11862,Father of the Bride Part II,"Comedy, Family",Just when George Banks has recovered from his ...,/content/posters/posters/11862.jpg


In [12]:
# Celda 6 — Definir SiglipEncoder y MovieIndex inline
import torch
import numpy as np
import faiss
from PIL import Image
from transformers import AutoProcessor, AutoModel
from tqdm import tqdm

print(f"GPU disponible: {torch.cuda.is_available()}")

MODEL_ID = "google/siglip-base-patch16-224"


class SiglipEncoder:
    def __init__(self, model_id=MODEL_ID, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Cargando SigLIP en {self.device}...")
        self.model = AutoModel.from_pretrained(model_id).to(self.device)
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model.eval()

    def _image_features(self, pixel_values):
        out = self.model.get_image_features(pixel_values=pixel_values)
        # get_image_features puede devolver un objeto o un tensor según la versión
        feats = out.pooler_output if hasattr(out, "pooler_output") else out
        return feats / feats.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def encode_images_batch(self, image_paths, batch_size=32):
        all_features = []
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Codificando pósters"):
            batch = image_paths[i : i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch]
            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
            feats = self._image_features(inputs["pixel_values"])
            all_features.append(feats.cpu().numpy())
        return np.vstack(all_features)


class MovieIndex:
    def __init__(self, dim=768):
        self.index = faiss.IndexFlatIP(dim)
        self.metadata = None

    def build(self, embeddings, metadata):
        self.index.add(embeddings.astype(np.float32))
        self.metadata = metadata.reset_index(drop=True)

    def save(self, index_path, metadata_path):
        faiss.write_index(self.index, index_path)
        self.metadata.to_csv(metadata_path, index=False)
        print(f"Índice guardado:   {index_path}")
        print(f"Metadata guardada: {metadata_path}")

GPU disponible: True


In [14]:
# Celda 7 — Encodear con SigLIP
encoder    = SiglipEncoder()
embeddings = encoder.encode_images_batch(df["poster_path"].tolist())

print(f"Shape embeddings: {embeddings.shape}")  # esperado: (N, 768)

Codificando pósters: 100%|██████████| 2659/2659 [29:55<00:00,  1.48it/s]


Shape embeddings: (85082, 768)
Cargando SigLIP en cuda...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Codificando pósters:   8%|▊         | 213/2659 [02:25<27:47,  1.47it/s]


KeyboardInterrupt: 

In [17]:
# Celda 8 — Construir y guardar el índice
INDEX_PATH = "/content/drive/MyDrive/Recomendador_Pelis/faiss_siglip.index"
META_PATH  = "/content/drive/MyDrive/Recomendador_Pelis/index_metadata_siglip.csv"

idx = MovieIndex(dim=embeddings.shape[1])
idx.build(embeddings, df)
idx.save(INDEX_PATH, META_PATH)

print(f"\nListo. {len(df):,} películas indexadas.")

# Backup: descarga directa al navegador
from google.colab import files
files.download(INDEX_PATH)
files.download(META_PATH)

Índice guardado:   /content/drive/MyDrive/Recomendador_Pelis/faiss_siglip.index
Metadata guardada: /content/drive/MyDrive/Recomendador_Pelis/index_metadata_siglip.csv

Listo. 85,082 películas indexadas.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# Celda 9 — Verificación rápida
sample = df.sample(1, random_state=42).iloc[0]

# Encodear la query
query_img = Image.open(sample["poster_path"]).convert("RGB")
inputs    = encoder.processor(images=query_img, return_tensors="pt").to(encoder.device)
with torch.no_grad():
    query_vec = encoder._image_features(inputs["pixel_values"]).cpu().numpy()

# Buscar vecinos
scores, indices = idx.index.search(query_vec.astype(np.float32), 5)
resultados = df.iloc[indices[0]]

print(f"Query: {sample['title']} ({sample['genres']})")
print()
for i, (_, row) in enumerate(resultados.iterrows()):
    print(f"  {i+1}. {row['title']} ({row['genres']})  — score: {scores[0][i]:.4f}")

Query: Violeta Doesn't Take the Elevator (Comedy, Romance)

  1. Violeta Doesn't Take the Elevator (Comedy, Romance)  — score: 1.0000
  2. Dayveon (Drama)  — score: 0.6376
  3. Sex and the Matrix (Comedy)  — score: 0.6327
  4. Isi & Ossi (Romance, Comedy)  — score: 0.6316
  5. Minutes (Comedy, Drama)  — score: 0.6308
